# Module 4 — Entity and Relationship Extraction with LLMs

**The gap, from Modules 1-3:** the graph so far holds what's *structured* — filings chunked into
text, plus executives/financials/news pulled from external sources. But the filing text itself is
full of entities and relationships that never get surfaced: named regulations, subsidiaries,
products, risk factors, all sitting inertly inside `Chunk.text`, invisible to Cypher.

**What we build in this module:**
- Mine chunk text with an LLM, and see why prompt design matters before committing to an
  approach: a bare "extract entities and relationships" prompt produces a different,
  inconsistent type vocabulary almost every run — not usable for a Cypher query that expects a
  `SUBSIDIARY_OF` edge to always be spelled `SUBSIDIARY_OF`
- Add a predefined type schema to fix the vocabulary problem, then confront the subtler issue it
  doesn't fix: asked to extract entities and relationships in the same breath, the model invents
  relationships to things it never firmly committed to as entities
- Split the process — settle the entity list first, double-check it with a reflection pass, *then*
  extract relationships constrained to that settled list — removing that class of hallucination
  by construction
- Ship that two-phase pipeline to `src/extraction/` and run it for real, writing generic
  `RecognisedEntity` nodes and `RELATED_TO` edges to Neo4j — deliberately not merged across
  chunks/documents or reconciled with the curated `Company`/`Person` nodes from Modules 1/3;
  that reconciliation is Module 5's job
- Add two new agent tools that query this extracted structure directly, and compare an agent
  with and without them on questions that need it

**New components introduced:**
- `extraction.validators` — `EntityType`/`RelationshipType` enums, `ExtractedEntity`/`ExtractedRelationship`
- `extraction.prompts` — role/scope preamble, entity/reflection/relationship prompts
- `extraction.entities` — `extract_entities()`: pass A (entities) + pass B (reflection)
- `extraction.relationships` — `extract_relationships()`: pass C (relationships, closed entity list); `write_extraction_to_graph()`
- `ingestion.schema.apply_extraction_schema()` — `RecognisedEntity(string, doc_id)` uniqueness constraint
- `retrieval.graph_nav.get_recognised_entities`/`get_entity_relationships` + matching `agent.tools` — query the extracted structure directly; bound as `MODULE_4_TOOLS` with `MODULE_4_STRATEGY_PROMPT`

> Run `scripts/run_extraction.py` to process the full corpus (long-running; this notebook only
> runs a small demo batch).

## 1. Picking chunks to work with

Not every chunk is worth extracting from — plenty of 10-K text is boilerplate (signature blocks,
certifications) with little to mine. We'll work with one chunk throughout the naive → rigorous →
two-phase comparison below: 3M's **Item 1. Business** overview, which is short enough to read in
full but already has a company, a jurisdiction, a regulator, a named law, and three business
segments packed into a few sentences — enough variety to see a prompt succeed or fail on more
than one entity type at once.

In [1]:
from langchain_core.documents import Document
from financial_advisor.services.neo4j_service import neo4j_service

BUSINESS_CHUNK_ID = "3M/3M_2025_10K.pdf/6"

row = neo4j_service.run_query(
    "MATCH (c:Chunk {id: $id}) RETURN c.text AS text", {"id": BUSINESS_CHUNK_ID}
)[0]
business_doc = Document(page_content=row["text"], metadata={"id": BUSINESS_CHUNK_ID})
print(business_doc.page_content)


Item 1. Business
3M Company was incorporated in 1929 under the laws of the State of Delaware to continue operations begun in 1902. The Company's ticker symbol is MMM. As used herein, the term '3M' or 'Company' includes 3M Company and its subsidiaries unless the context indicates otherwise. In this document, for any references to Note 1 through Note 20, refer to the Notes to Consolidated Financial Statements in Item 8.
Available Information : The Securities and Exchange Commission (SEC) maintains a website that contains reports, proxy and information statements, and other information regarding issuers, including the Company, that file electronically with the SEC. The public can obtain any documents that the Company files with the SEC at https://www.sec.gov. The Company files annual reports, quarterly reports, proxy statements and other documents with the SEC under the Securities Exchange Act of 1934 (Exchange Act).
3M also makes available free of charge through its website (https://inve

## 2. A naive prompt

The most obvious approach: ask the model to "extract entities and relationships," give it a
loose schema (a free-text `type` field, no constraints), and see what comes back. No role, no
scope, no predefined vocabulary — just the instruction and the text.

In [2]:
# Ad hoc, throwaway schema — deliberately NOT the one in extraction.validators. This whole cell
# exists to be visibly worse than what ships in src/, so it stays notebook-only (see adr/0007).
from pydantic import BaseModel, Field

from financial_advisor.clients import get_llm


class NaiveEntity(BaseModel):
    name: str
    type: str
    description: str | None = None


class NaiveRelationship(BaseModel):
    source: str
    target: str
    type: str


class NaiveResult(BaseModel):
    entities: list[NaiveEntity] = Field(default_factory=list)
    relationships: list[NaiveRelationship] = Field(default_factory=list)


naive_prompt = (
    "Extract entities and relationships from this text, focusing on things relevant to "
    f"financial analysis:\n\n{business_doc.page_content}"
)
naive_result = get_llm().with_structured_output(NaiveResult).invoke(naive_prompt)

for e in naive_result.entities:
    print(f"{e.type:25s} | {e.name}")
print()
for r in naive_result.relationships:
    print(f"{r.source} -[{r.type}]-> {r.target}")

Company                   | 3M Company
Ticker                    | MMM
Jurisdiction              | State of Delaware
Regulator                 | Securities and Exchange Commission (SEC)
Website                   | sec.gov
Website                   | investors.3M.com
SEC Filing                | Form 10-K
SEC Filing                | Form 10-Q
SEC Filing                | Form 8-K
Law                       | Securities Exchange Act of 1934
Business Segment          | Safety and Industrial
Business Segment          | Transportation and Electronics
Business Segment          | Consumer
Financial Note            | Notes to Consolidated Financial Statements (Item 8)
Corporate Structure       | Subsidiaries
Competitor Group          | Competitors (technologically oriented companies)

3M Company -[incorporated_in]-> State of Delaware
3M Company -[has_ticker]-> MMM
3M Company -[has_subsidiaries]-> Subsidiaries
3M Company -[operates_segment]-> Safety and Industrial
3M Company -[operates_segment]-> 

**What went wrong.** Run this cell and look at the `type` column: in one run it came back
`Ticker`, `Jurisdiction`, `Regulatory Agency`, `Website`, `Filing Type`, `Business Segment`,
`Subsidiaries`, `Competitors (general)` — eight ad hoc types for eleven entities, none of them
chosen from any fixed vocabulary because none was given. Run the cell again and you'll likely get
a *different* eight. That's fatal for a graph you intend to query: `MATCH (n:RecognisedEntity
{type: "Regulation"})` only works if "Regulation" is spelled the same way every time it's
produced, and nothing here enforces that.

The relationships are worse. That one run produced 22 relationship edges over `incorporated_in`,
`operations_began`, `has_ticker`, `includes`, `files_reports_with`, `hosts_public_filings_on`,
`posts_filings_on`, `files_under_regulation`, `files_document_type` (×5), `operates_in_segment`
(×3), `faces_competition_from`, and `provides_document_type` (×5) — a different invented verb
for nearly every edge, several of them near-duplicates of each other (`files_document_type` and
`provides_document_type` say almost the same thing from two different sources). Instructing the
model to "focus on things relevant to financial analysis" did not stop it from extracting the
`https://www.sec.gov` URL as a `Website` entity. The instruction was too underspecified to
constrain either the type vocabulary or what counts as worth extracting.

## 3. Adding Rigor: Role, Scope, Predefined Types

Second attempt: give the model a role ("financial analyst"), a scope (this is a 10-K, extract
only what's explicitly stated), and a closed type vocabulary for both entities and
relationships — the actual `EntityType`/`RelationshipType` enums from `extraction.validators`.
Still one call for entities *and* relationships together, same as the naive version — only the
type discipline changes.

In [ ]:
from financial_advisor.extraction.validators import EntityType, RelationshipType


class SingleShotEntity(BaseModel):
    string: str
    type: EntityType


class SingleShotRelationship(BaseModel):
    source: str
    target: str
    type: RelationshipType


class SingleShotResult(BaseModel):
    entities: list[SingleShotEntity] = Field(default_factory=list)
    relationships: list[SingleShotRelationship] = Field(default_factory=list)


entity_types = ", ".join(t.value for t in EntityType)
relationship_types = ", ".join(t.value for t in RelationshipType)
single_shot_prompt = f"""\
You are a financial analyst extracting structured knowledge from a company's SEC 10-K filing.
Extract entities of these types only: {entity_types}.
Extract relationships of these types only: {relationship_types}. Source/target must be entities
you extracted. Only extract what the text explicitly states.

Text:
{business_doc.page_content}"""

single_shot_result = get_llm().with_structured_output(SingleShotResult).invoke(single_shot_prompt)

for e in single_shot_result.entities:
    print(f"{e.type.value:20s} | {e.string}")
print()
for r in single_shot_result.relationships:
    print(f"{r.source} -[{r.type.value}]-> {r.target}")

**Better, but not fixed.** The type vocabulary is now consistent — every entity and relationship
is spelled from the fixed enum, run after run. But a second failure mode shows up. In one run,
the model extracted **"Other technologically oriented companies"** — a vague noun phrase from the
sentence "products... are subject to competition from products manufactured and sold by other
technologically oriented companies" — as a `Company` entity, then confidently linked it with
`3M Company -[COMPETES_WITH]-> Other technologically oriented companies`. That's not a real
competitor; it's a category description turned into a relationship target because the model
extracted entities and relationships in the same breath and never had to commit to "is this
actually a company?" before using it as one. The same run separately extracted "Competition"
itself as a `Risk` entity with an `EXPOSED_TO` edge — two different, overlapping ways of encoding
the same one sentence. Fixing the type vocabulary didn't fix the model inventing relationships to
things it never firmly settled on as entities.

## 4. Splitting the Process: Entities → Reflection → Relationships

Third attempt, and the one that ships to `src/extraction/`: stop asking for entities and
relationships in one call. Settle the entity list first — a first pass, then a second
"anything missed?" reflection pass shown the text plus its own first-pass list — and only once
that list is settled, extract relationships constrained to name only entities from it. This is
`extract_entities()` and `extract_relationships()` from `extraction.entities`/
`extraction.relationships`, not a notebook throwaway — the real implementation.

In [5]:
from financial_advisor.extraction.entities import extract_entities

entity_result = extract_entities(business_doc)
for e in entity_result.entities:
    print(f"{e.type.value:20s} | {e.string}")


[extract-entities] pass A: 21 entit(y/ies) found
[extract-entities] reflection: 7 addition(s)/correction(s)
Company              | 3M Company
Company              | 3M
Company              | Company
Company              | MMM
Location             | State of Delaware
Regulation           | Securities and Exchange Commission
Regulation           | SEC
Location             | https://www.sec.gov
Regulation           | Securities Exchange Act of 1934
Regulation           | Exchange Act
Location             | https://investors.3M.com
Product              | Annual Report on Form 10-K
Product              | Quarterly Reports on Form 10-Q
Product              | Current Reports on Form 8-K
Product              | Form 10-K
Product              | Form 10-Q
Product              | Form 8-K
Product              | Safety and Industrial
Product              | Transportation and Electronics
Product              | Consumer
Product              | Notes to Consolidated Financial Statements in Item 8
Produc

Watch the `[extract-entities]` log lines above: pass A typically finds around 9 entities —
including, tellingly, *both* "3M Company" and the short form "3M" as separate entities (they're
different strings, so under the `(string, doc_id)` key they stay separate — a real limitation
this design accepts and defers to Module 5's resolution pass, not a bug). The reflection pass
then adds a handful more, commonly the specific filing types (Form 10-K, 10-Q, 8-K) that pass A
named collectively but didn't break out individually. The two URLs in the text are a good
illustration of a closed vocabulary's cost: nothing in `EntityType` actually fits "web address,"
so a URL like this can get force-typed as `Location` by the reflection pass — the closest
available type, and an honest cost of the closed vocabulary rather than a bug in it. It stops
the type chaos from section 2, but it will occasionally force-fit something that doesn't belong
anywhere. A vocabulary gap like that is a real refinement to make to `EntityType`, not a reason
to abandon having one.

In [6]:
from financial_advisor.extraction.relationships import extract_relationships

known_entity_names = sorted({e.resolved_string for e in entity_result.entities})
relationship_result = extract_relationships(business_doc, known_entity_names)
for r in relationship_result.relationships:
    print(f"{r.source} -[{r.type.value}]-> {r.target}")


[extract-relationships] 13 relationship(s) found
3M Company -[OPERATES_IN]-> Safety and Industrial
3M Company -[OPERATES_IN]-> Transportation and Electronics
3M Company -[OPERATES_IN]-> Consumer
3M Company -[SUPPLIES]-> Annual Report on Form 10-K
3M Company -[SUPPLIES]-> Quarterly Reports on Form 10-Q
3M Company -[SUPPLIES]-> Current Reports on Form 8-K
3M Company -[SUPPLIES]-> proxy statements
3M Company -[REGULATED_BY]-> Securities Exchange Act of 1934
3M Company -[OPERATES_IN]-> https://investors.3M.com
https://investors.3M.com -[SUPPLIES]-> Annual Report on Form 10-K
Securities and Exchange Commission -[OPERATES_IN]-> https://www.sec.gov
https://www.sec.gov -[SUPPLIES]-> proxy statements
https://www.sec.gov -[SUPPLIES]-> information statements


Notice what's missing compared to section 3: no `COMPETES_WITH` edge to "other technologically
oriented companies," because that phrase never made it into the settled entity list in the first
place — `extract_relationships` only offers the model names that were already confirmed to exist,
so there's nothing for it to hallucinate a relationship onto. `_filter_valid_relationships`
(`extraction/relationships.py`) is the backstop for the rest: it drops any relationship whose
source or target isn't in the known list, in case the model names something anyway despite the
prompt constraint, and logs how many it dropped. Splitting entities from relationships didn't
just make the output cleaner — it removed an entire class of hallucination by construction,
rather than trying to prompt it away.

## 4a. A Refinement: Resolving Coreference

Look back at the very first chunk in section 1 — its own text defines the term this filing will
use for itself: *"As used herein, the term '3M' or 'Company' includes 3M Company and its
subsidiaries unless the context indicates otherwise."* From that point on, much of the filing
refers to 3M not by name but as **"the Company"** — a defined term, not a generic pronoun, so it
reads as a legitimate named entity to extract.

Extracted literally, every one of those mentions becomes its own entity, disconnected from the
one named "3M". Without this section's fix, a run over the Risk Factors chunk below can produce
output like this:

```
(Company) The Company -[EXPOSED_TO]-> (Risk) AFFF multi-district litigation
(Company) The Company -[EXPOSED_TO]-> (Risk) PWS Settlement
(Company) 3M -[EXPOSED_TO]-> (FinancialMetric) $10.5 billion to $12.5 billion in total
```

Same company, same filing, same kind of relationship — split across two unconnected identities
just because one mention happened to be phrased as a name and another as a defined term.

**Why this isn't Module 5's job.** Module 5 (similarity/entity resolution, coming up) reconciles
extracted mentions by *name similarity* — fuzzy string matching narrows candidates before an LLM
judges whether they're the same thing. "the Company" and "3M" share no common substring, so "3M"
would never even become a candidate for "the Company" to be judged against — the pair never
reaches the LLM judge at all. Module 5 is the right tool for genuinely different name strings
("Solventum Corporation" vs "Solventum"); it structurally can't catch a defined-term
self-reference.

**Where coreference gets resolved.** Reflection (pass B) already receives the settled
first-pass entity list *and* the source text side by side — exactly the two things needed to spot
*any* coreference, not just a filer referring to itself as "the Company". `ExtractedEntity`
keeps two fields — `string` (the literal text, e.g. "the Company", never discarded) and
`resolved_string` (the canonical entity this mention refers to) — and resolving them is
reflection's job: given the entities already found and the chunk text as evidence, check whether
any entity is really just another way of referring to a *different* entity already in the list,
and if so, report `resolved_string` as that other entity's own `string`. This generalizes well
beyond the "the Company" case — the same mechanism can resolve abbreviations to their spelled-out
forms too, for example:

```
Company     | string='The Company' -> resolved_string='3M'
Location    | string='U.S.' -> resolved_string='United States'
Regulation  | string='CERCLA' -> resolved_string="United States Comprehensive Environmental
              Response, Compensation and Liability Act of 1980 ('CERCLA')"
Regulation  | string='EPA' -> resolved_string='U.S. Environmental Protection Agency (EPA)'
```

**The backstop.** A natural-language instruction is never 100% reliable — this notebook has
already made that point twice (sections 2 and 3) — so `resolved_string` still gets checked in
code, not just asked for nicely, the same way `_filter_valid_relationships` already checks
relationships. `extraction.entities._restrict_resolved_strings_to_settled_entities` resets
`resolved_string` back to `string` whenever it doesn't match another entity's `string` that's
actually in the settled list *and* of the same type — catching both an invented canonical form
nobody extracted, and a cross-type mix-up (a `Risk` sentence merely mentioning "the Company"
getting redirected to a `Company` entity's string).

**A trade-off worth knowing about.** Resolving coreference this way can only resolve a mention to
an entity that's actually extracted somewhere in the *same* chunk; if "3M" itself isn't a settled
entity in a given chunk, "the Company" there has nothing to resolve to and stays
self-referential. In practice a chunk that talks about "the Company" at length also names the
company directly at least once, so this is expected to be rare.

In [7]:
from financial_advisor.extraction.entities import extract_entities
from financial_advisor.extraction.relationships import extract_relationships

COREF_CHUNK_ID = "3M/3M_2025_10K.pdf/17"  # Risk Factors — Legal and Regulatory Proceedings (PFAS)

row = neo4j_service.run_query(
    "MATCH (c:Chunk {id: $id}) RETURN c.text AS text", {"id": COREF_CHUNK_ID}
)[0]
coref_doc = Document(page_content=row["text"], metadata={"id": COREF_CHUNK_ID})

coref_entities = extract_entities(coref_doc).entities
print("Entities where the literal text differs from the resolved identity:")
for e in coref_entities:
    if e.string != e.resolved_string:
        print(f"  {e.type.value:10s} | string={e.string!r:20s} -> resolved_string={e.resolved_string!r}")

known_coref_entities = sorted({e.resolved_string for e in coref_entities})
coref_relationships = extract_relationships(coref_doc, known_coref_entities).relationships
print("\nRelationships (source/target are now the resolved identity):")
for r in coref_relationships:
    print(f"  {r.source} -[{r.type.value}]-> {r.target}")


[extract-entities] pass A: 34 entit(y/ies) found
[extract-entities] reflection: 4 addition(s)/correction(s)
Entities where the literal text differs from the resolved identity:
  Company    | string='The Company'        -> resolved_string='3M'
[extract-relationships] 21 relationship(s) found

Relationships (source/target are now the resolved identity):
  perfluoroalkyl and polyfluoroalkyl substances, collectively known as 'PFAS,' -[REGULATED_BY]-> United States
  3M -[PRODUCES]-> perfluoroalkyl and polyfluoroalkyl substances, collectively known as 'PFAS,'
  3M -[PRODUCES]-> perfluorooctanoate (PFOA)
  3M -[PRODUCES]-> perfluorooctane sulfonate (PFOS)
  3M -[PRODUCES]-> Aqueous Film Forming Foam (AFFF)
  3M -[REGULATED_BY]-> United States Comprehensive Environmental Response, Compensation and Liability Act of 1980 ('CERCLA')
  perfluorooctanoate (PFOA) -[REGULATED_BY]-> United States Comprehensive Environmental Response, Compensation and Liability Act of 1980 ('CERCLA')
  perfluorooctane

## 5. Storage: `RecognisedEntity` and `RELATED_TO`

Two more decisions:

- **One generic node label, `RecognisedEntity`**, not per-type labels like `:Company`/`:Person`.
  Real labels would collide with the *curated* `Company`/`Person` nodes Modules 1/3 already
  populate — an extracted mention isn't confirmed to be the same node as the curated one yet
  (that reconciliation is Module 5). `type` lives as a property instead. The uniqueness key is
  `(string, doc_id)` — same string mentioned twice in the same document merges into one node;
  the same string in a different document, or a near-miss like "3M" vs. "3M Company," stays
  separate. No cross-document merging yet, by design.
- **One generic edge, `RELATED_TO {type, evidence, chunk_id}`**, not dynamic edge types matching
  the relationship enum. Same collision reasoning — a `RecognisedEntity-[:SUBSIDIARY_OF]->
  RecognisedEntity` edge would sit right next to the *real* curated `Company-[:SUBSIDIARY_OF]->
  Company` edges from Module 3's Wikidata structure data, and it's not obvious which is which
  without checking the node labels. It also means Module 5's relationship reconciliation scans
  one edge type instead of nine.

Provenance is `(:RecognisedEntity)-[:MENTIONED_IN]->(:Chunk)`, so every extracted node traces back
to the exact text it came from. `string` on a `RecognisedEntity` is the *resolved* identity (see
4a above) — the `(string, doc_id)` uniqueness key is applied after coreference resolution, not
before, so "the Company" and "3M" mentions already collapse onto one node by the time this key
matters. Every distinct literal form actually seen lands in a separate `mentions` list property.

In [8]:
from financial_advisor.ingestion.schema import apply_extraction_schema

apply_extraction_schema()

  [schema] OK  recognised_entity_key
[schema] 1/1 statements applied — all good


## 6. Running the Pipeline for Real

A small batch — the Business Overview chunk from above plus three more picked for variety: a
dense Legal/Regulatory Risk Factors passage (PFAS litigation — lots of named substances and
dollar figures), the MD&A Overview (names 3M's 2024 spin-off of Solventum Corporation and a
financial-highlights table), and a subsidiaries table from the 2024 filing (clean, unambiguous
`SUBSIDIARY_OF`-shaped data — every row is `(company name, jurisdiction)`). Same
`extract_entities` → `extract_relationships` → `write_extraction_to_graph` pipeline as above, now
actually writing to Neo4j. The full corpus is `scripts/run_extraction.py`'s job, not this
notebook's — this is a few chunks so the result is visible in one run.

In [9]:
from financial_advisor.extraction.relationships import write_extraction_to_graph

DEMO_CHUNK_IDS = [
    BUSINESS_CHUNK_ID,
    "3M/3M_2025_10K.pdf/17",  # Risk Factors — Legal and Regulatory Proceedings (PFAS)
    "3M/3M_2025_10K.pdf/33",  # MD&A Overview (Solventum separation, financial highlights)
    "3M/3M_2024_10K.pdf/244",  # 3M Company and Consolidated Subsidiaries
]

rows = neo4j_service.run_query(
    "MATCH (c:Chunk) WHERE c.id IN $ids RETURN c.id AS id, c.text AS text, c.doc_id AS doc_id",
    {"ids": DEMO_CHUNK_IDS},
)

for i, row in enumerate(rows, start=1):
    print(f"[{i}/{len(rows)}] chunk {row['id']}")
    doc = Document(page_content=row["text"], metadata={"id": row["id"]})

    entities = extract_entities(doc).entities
    known_entities = sorted({e.resolved_string for e in entities})
    relationships = extract_relationships(doc, known_entities).relationships
    write_extraction_to_graph(entities, relationships, chunk_id=row["id"], doc_id=row["doc_id"])
    neo4j_service.run_query("MATCH (c:Chunk {id: $id}) SET c.extracted = true", {"id": row["id"]})
    print(f"    wrote {len(entities)} entities, {len(relationships)} relationships")


[1/4] chunk 3M/3M_2025_10K.pdf/6
[extract-entities] pass A: 10 entit(y/ies) found
[extract-entities] reflection: 6 addition(s)/correction(s)
[extract-relationships] dropped 1 relationship(s) referencing unknown entities
[extract-relationships] 8 relationship(s) found
    wrote 14 entities, 8 relationships
[2/4] chunk 3M/3M_2025_10K.pdf/17
[extract-entities] pass A: 34 entit(y/ies) found
[extract-entities] reflection: 4 addition(s)/correction(s)
[extract-relationships] 15 relationship(s) found
    wrote 36 entities, 15 relationships
[3/4] chunk 3M/3M_2025_10K.pdf/33
[extract-entities] pass A: 34 entit(y/ies) found
[extract-entities] reflection: 5 addition(s)/correction(s)
[extract-relationships] 16 relationship(s) found
    wrote 37 entities, 16 relationships
[4/4] chunk 3M/3M_2024_10K.pdf/244
[extract-entities] pass A: 55 entit(y/ies) found
[extract-entities] reflection: 0 addition(s)/correction(s)
[extract-relationships] 35 relationship(s) found
    wrote 55 entities, 35 relationships

In [10]:
rows = neo4j_service.run_query(
    """
    MATCH (a:RecognisedEntity)-[r:RELATED_TO]->(b:RecognisedEntity)
    RETURN a.string AS source, a.type AS source_type, r.type AS relationship,
           b.string AS target, b.type AS target_type
    ORDER BY relationship
    LIMIT 25
    """
)
for row in rows:
    print(f"({row['source_type']}) {row['source']} -[{row['relationship']}]-> "
          f"({row['target_type']}) {row['target']}")

(FinancialMetric) Operating income margin -[EXPOSED_TO]-> (Product) manufactured PFAS products
(FinancialMetric) Operating income margin -[EXPOSED_TO]-> (Risk) cost dis-synergies
(FinancialMetric) Operating income margin -[EXPOSED_TO]-> (Risk) site remediation obligations
(FinancialMetric) Operating income margin -[EXPOSED_TO]-> (Risk) 2025 PFAS-related New Jersey Settlement
(FinancialMetric) Earning per diluted share (EPS) -[EXPOSED_TO]-> (FinancialMetric) $0.8 billion pre-tax pension settlement charge in 2024
(Company) 3M -[EXPOSED_TO]-> (FinancialMetric) $10.5 billion to $12.5 billion in total
(Risk) significant litigation -[EXPOSED_TO]-> (Risk) 2025 PFAS-related New Jersey Settlement
(FinancialMetric) Earning per diluted share (EPS) -[EXPOSED_TO]-> (Risk) imputed interest associated with obligations resulting from significant litigation
(FinancialMetric) Earning per diluted share (EPS) -[EXPOSED_TO]-> (Company) Solventum Corporation (Solventum)
(Company) 3M -[EXPOSED_TO]-> (Risk) l

The subsidiaries table is worth checking on its own — it's the chunk with the clearest ground
truth (every row *should* produce one `Company -[SUBSIDIARY_OF]-> Company` edge with a
`Location`-typed jurisdiction alongside it), so it's the easiest one to sanity-check the pipeline
against.

In [11]:
rows = neo4j_service.run_query(
    """
    MATCH (e:RecognisedEntity)-[:MENTIONED_IN]->(:Chunk {id: "3M/3M_2024_10K.pdf/244"})
    OPTIONAL MATCH (e)-[r:RELATED_TO {type: "SUBSIDIARY_OF"}]->(parent:RecognisedEntity)
    RETURN e.string AS entity, e.type AS type, parent.string AS parent_of
    ORDER BY type, entity
    """
)
for row in rows:
    print(f"{row['type']:10s} | {row['entity']:45s} | SUBSIDIARY_OF -> {row['parent_of']}")

Company    | 3M Belgium BV                                 | SUBSIDIARY_OF -> 3M Company
Company    | 3M Canada Company - Compagnie 3M Canada       | SUBSIDIARY_OF -> 3M Company
Company    | 3M Chemical Operations LLC                    | SUBSIDIARY_OF -> 3M Company
Company    | 3M China Limited                              | SUBSIDIARY_OF -> 3M Company
Company    | 3M Company                                    | SUBSIDIARY_OF -> None
Company    | 3M Deutschland GmbH                           | SUBSIDIARY_OF -> 3M Company
Company    | 3M EMEA GmbH                                  | SUBSIDIARY_OF -> 3M Company
Company    | 3M Fall Protection Company                    | SUBSIDIARY_OF -> 3M Company
Company    | 3M Financial Management Company               | SUBSIDIARY_OF -> 3M Company
Company    | 3M Foreign Holding LLC                        | SUBSIDIARY_OF -> 3M Company
Company    | 3M France S.A.S.                              | SUBSIDIARY_OF -> 3M Company
Company    | 3M Global Capi

### Check the database
Let's check our database using the browser at [http://localhost:7474/browser/](http://localhost:7474/browser/)

## 7. New Questions This Structure Unlocks

Everything above justified the pipeline in the abstract. Here's the payoff: two new agent tools,
`get_recognised_entities`/`get_entity_relationships` (`retrieval/graph_nav.py`,
`agent/tools.py`), that query `RecognisedEntity`/`RELATED_TO` directly instead of asking the
model to re-derive structure from raw chunk text on every question. They're bound alongside
Module 3's tools as `MODULE_4_TOOLS`, with `MODULE_4_STRATEGY_PROMPT` telling the strategy agent
when to reach for them.

We'll ask the *same* two questions of two agents — one built with `MODULE_3_TOOLS` (no access to
the extracted structure, has to work from raw chunk text), one with `MODULE_4_TOOLS` — and
compare. Both questions are chosen to be hard specifically *because* the answer requires
aggregating over the full subsidiaries table, not because the underlying text is hidden or
missing.

In [12]:
from financial_advisor.agent.graph import build_agent
from financial_advisor.agent.prompts import MODULE_3_STRATEGY_PROMPT, MODULE_4_STRATEGY_PROMPT
from financial_advisor.agent.state import initial_state
from financial_advisor.agent.tools import MODULE_3_TOOLS, MODULE_4_TOOLS

agent_without = build_agent(MODULE_3_TOOLS, MODULE_3_STRATEGY_PROMPT)
agent_with = build_agent(MODULE_4_TOOLS, MODULE_4_STRATEGY_PROMPT)


def ask(agent, question: str) -> dict:
    result = agent.invoke(initial_state(question), {"recursion_limit": 50})
    tool_sequence = [entry["tool"] for entry in result["tool_call_log"]]
    print(f"[{result['retrieval_iterations']} retrieval round(s)] tools called: {tool_sequence}")
    print(f"\nA: {result['answer']}")
    return result

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

### 7a. "What subsidiaries are mentioned, and where?"

The table itself is fully inside one chunk (`3M/3M_2024_10K.pdf/244`), so a text search *can*
retrieve the right raw material in one shot. The question is what each agent does with it.

In [13]:
Q1 = (
    "What are the subsidiary companies mentioned in 3M's 2024 10-K (doc_id "
    "3M/3M_2024_10K.pdf), and in which countries/jurisdictions are they organized?"
)

print("===== WITHOUT the new tools =====")
result_without_q1 = ask(agent_without, Q1)

===== WITHOUT the new tools =====
[strategy] iteration 1: 1 tool call(s) planned
    - fulltext_search({'query': 'subsidiaries OR "Principal subsidiaries" OR "Principal Subidiaries" OR "significant subsidiaries"', 'k': 10, 'company_id': '3M', 'year': 2024})
[tools] fulltext_search({'query': 'subsidiaries OR "Principal subsidiaries" OR "Principal Subidiaries" OR "significant subsidiaries"', 'k': 10, 'company_id': '3M', 'year': 2024}) -> 10 chunk(s)
[grade-retrieval] sufficient=False
    feedback: Not sufficient — the retrieved chunk appears to be only part of the "3M Company and Consolidated Subsidiaries" table and likely continues on adjacent pages. To produce a complete, precise answer listing every subsidiary named in the 2024 10-K and the jurisdiction for each, fetch the full table from the filing. Use the get_document_pages tool for doc_id=3M/3M_2024_10K.pdf and request the pages that contain the full table (start with pages 168–172 or 166–174 to be safe). Alternatively, run fullte

In [14]:
print("===== WITH the new tools =====")
result_with_q1 = ask(agent_with, Q1)

===== WITH the new tools =====
[strategy] iteration 1: 1 tool call(s) planned
    - get_recognised_entities({'doc_id': '3M/3M_2024_10K.pdf', 'entity_type': 'Company'})
[tools] get_recognised_entities({'doc_id': '3M/3M_2024_10K.pdf', 'entity_type': 'Company'}) -> 36 chunk(s)
[grade-retrieval] sufficient=False
    feedback: Missing: the jurisdiction (country/organized-under-law) entries for each subsidiary listed in the schedule of subsidiaries in 3M's 2024 10-K. To complete the answer, retrieve the schedule rows showing the "organized under the laws of" column. Recommended next calls (use company_id="3M" and year="2024"):

1) fulltext_search with a phrase likely present in the subsidiary schedule, e.g. query: "organized under the laws of" (or try "organized under the law" / "Organized under") with company_id="3M" and year="2024". This should surface the chunks containing the subsidiary table including the jurisdiction column.

2) If fulltext_search returns a promising chunk/doc_id+page 

**A run without the extracted structure — what this commonly looks like:** the *without* agent's
first `semantic_search` call can already retrieve the complete subsidiaries table — but the
retrieval-grading step, looking only at raw chunk text with no way to confirm the table had
ended, tends to suspect there must be more pages and ask for them. That sends the agent chasing
a "continuation" that doesn't exist: more search calls, then `get_document_pages` on invented
follow-up pages, burning through retrieval rounds it didn't need. It can land on the *correct*
subsidiary list almost by accident, yet still hedge the final answer — something like "I need
the remainder... if you can provide or allow retrieval of the next pages, I will produce the
full list" — asking the user to supply something that was already retrieved in round 1.

The *with* agent, by contrast, can call `get_recognised_entities` and `get_entity_relationships`
immediately, get the full subsidiaries structure in one round, do one confirming
`fulltext_search` pass, and answer directly — no hedging, no phantom missing pages, far fewer
rounds than the *without* agent needs. Same underlying facts, same source document; the
difference is that a structured, countable result set doesn't trigger the "is this really
everything?" doubt that raw table text does.

### 7b. "How many, and across how many countries?"

A harder question on purpose: it needs an aggregate (count non-US subsidiaries, count *distinct*
countries) over the same subsidiaries table, not just a transcription of it. The Cypher cell
below computes the ground truth directly against the graph built in section 6 — in one run it
came out to **35 total subsidiaries, 26 organized outside the US (9 under Delaware/Minnesota
law), across 17 distinct non-US countries**. Extraction is LLM-driven, so a different run over
the same chunk can land on slightly different counts — treat the numbers here as illustrative
rather than something to expect exactly.

In [15]:
Q2 = (
    "Per 3M's 2024 10-K (doc_id 3M/3M_2024_10K.pdf), how many of 3M's consolidated "
    "subsidiaries are organized outside the United States, and across how many distinct "
    "countries?"
)

print("===== WITHOUT the new tools =====")
result_without_q2 = ask(agent_without, Q2)

===== WITHOUT the new tools =====
[strategy] iteration 1: 1 tool call(s) planned
    - semantic_search({'query': 'consolidated subsidiaries organized outside the United States number distinct countries consolidated subsidiaries organized outside the U.S. how many countries "consolidated subsidiaries" "organized outside"', 'k': 10, 'company_id': '3M', 'year': 2024})
[tools] semantic_search({'query': 'consolidated subsidiaries organized outside the United States number distinct countries consolidated subsidiaries organized outside the U.S. how many countries "consolidated subsidiaries" "organized outside"', 'k': 10, 'company_id': '3M', 'year': 2024}) -> 10 chunk(s)
[grade-retrieval] sufficient=False
    feedback: The current extraction contains only one page (page 169) of the consolidated-subsidiaries listing from the 2024 10-K. It is likely the full consolidated-subsidiaries table spans additional page(s) in the filing. To answer precisely you must (1) retrieve the complete consolidated

In [16]:
print("===== WITH the new tools =====")
result_with_q2 = ask(agent_with, Q2)

===== WITH the new tools =====
[strategy] iteration 1: 2 tool call(s) planned
    - get_entity_relationships({'doc_id': '3M/3M_2024_10K.pdf', 'relationship_type': 'SUBSIDIARY_OF'})
    - get_entity_relationships({'doc_id': '3M/3M_2024_10K.pdf', 'relationship_type': 'OPERATES_IN'})
[tools] get_entity_relationships({'doc_id': '3M/3M_2024_10K.pdf', 'relationship_type': 'SUBSIDIARY_OF'}) -> 35 chunk(s)
[tools] get_entity_relationships({'doc_id': '3M/3M_2024_10K.pdf', 'relationship_type': 'OPERATES_IN'}) -> 0 chunk(s)
[grade-retrieval] sufficient=False
    feedback: Missing precisely what the question asks for: (a) the total number of consolidated subsidiaries that are organized outside the United States, and (b) the number of distinct countries those non‑U.S. subsidiaries are organized in. To close this gap with the available tools, try one of the following (do not repeat the same call already done):

1) Call get_recognised_entities with doc_id = "3M/3M_2024_10K.pdf" and entity_type = Comp

**A run without the extracted structure — what this commonly looks like:** the *without* agent
tends to burn through its retrieval rounds (`semantic_search`, `fulltext_search` variants,
`get_document_pages`) trying to locate an explicit summary sentence stating the counts —
because counting rows and their distinct values by eye from retrieved text isn't something the
retrieval-grading step trusts itself to do reliably, so it keeps asking for "the complete
listing" instead of counting what it already has. It's not unusual for it to end honestly
rather than guess, e.g. *"I cannot determine those numbers from the material retrieved."* That's
the *safe* failure — no fabricated count — but still a real capability gap: the document does
contain everything needed to answer.

The *with* agent pulls the entities and relationships directly, does one confirming search, and
can answer, for example: **"26 of 3M's consolidated subsidiaries are organized outside the
United States (35 total... minus 9 organized under U.S. law)... across 17 distinct foreign
countries."** When it lands on numbers like this, they match the ground truth computed in 7b
above. The aggregation happens where it's cheap and reliable — over structured rows — not
inside an LLM eyeballing a flattened table and trying to count.

### 7c. The Pattern

Both questions were answerable in principle from the raw text — nothing was hidden. What changed
is where the counting/completeness burden sits. Without structure, that burden falls on an LLM
reasoning over retrieved text snippets, and it shows up as two different failure modes: false
self-doubt about completeness (7a — burns retries chasing pages that don't exist) or an honest
refusal to count (7b — safe, but unhelpful). With `RecognisedEntity`/`RELATED_TO` in place, the
same questions become Cypher's job — exact, fast, and confident. That's the concrete payoff of
Module 4's extraction work, not just a cleaner-looking graph.

## 8. Where This Leaves the Graph

Four chunks, one consistent pipeline: in this run, the subsidiaries table alone produced 35
`Company` entities correctly `SUBSIDIARY_OF` the parent, plus 19 `Location` entities linked via
`OPERATES_IN` — extraction is LLM-driven, so a different run over the same chunk can land on a
slightly different count, but structurally: no stray types, no invented relationship verbs, no
relationship pointing at something that was never confirmed to exist. The PFAS and MD&A chunks
are messier (dense, narrative text always will be), but every entity and relationship in the
graph is at least spelled from the same fixed vocabulary and traceable to the chunk it came
from.

What's still true of everything just written: "3M" and "3M Company" are two different
`RecognisedEntity` nodes, and nothing here reconciles them with the curated `Company` nodes from
Module 1/3, even though they obviously refer to the same company. That's Module 5's entity
resolution — this module's job was only to get from unstructured text to a consistent,
queryable, honestly-labeled *first draft* of structure. Run `scripts/run_extraction.py` to
process the rest of the corpus once you're ready to move on.